In [ ]:
import json
from pathlib import Path

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

path = root / "data" / "processed" / "pages.jsonl"
pages = [json.loads(line) for line in open(path, encoding="utf-8")]

print("Total pages:", len(pages))
print("Keys in each record:", list(pages[0].keys()))

In [ ]:
for name in ["bhutan", "maldives", "nepal", "who"]:
    page = next(p for p in pages if p["doc"].startswith(name) and p["page"] == 12)
    print(page["doc"], "- page", page["page"])
    print(page["text"][:700])
    print()

In [ ]:
import re
from collections import Counter, defaultdict


def norm(line):
    return re.sub(r"\s+", " ", line).strip()


by_doc = defaultdict(list)
for p in pages:
    by_doc[p["doc"]].append(p)

# find lines that repeat on many pages of the same document
repeated = {}
for doc, ps in by_doc.items():
    counts = Counter()
    for p in ps:
        for line in set(norm(l) for l in p["text"].splitlines() if norm(l)):
            counts[line] += 1
    repeated[doc] = {line for line, c in counts.items() if c >= 0.3 * len(ps)}

page_no = re.compile(r"^\d{1,3}$|^[ivxlc]{1,6}$", re.IGNORECASE)
page_label = re.compile(r"^Page \d+$", re.IGNORECASE)


def clean(p):
    out = []
    for line in p["text"].splitlines():
        n = norm(line)
        if not n or n in repeated[p["doc"]]:
            continue
        if page_no.match(n) or page_label.match(n):
            continue
        out.append(n)
    return "\n".join(out)


cleaned = [{**p, "text": clean(p)} for p in pages]

print("Characters before:", sum(len(p["text"]) for p in pages))
print("Characters after: ", sum(len(p["text"]) for p in cleaned))
print()
page = next(p for p in cleaned if p["doc"].startswith("bhutan") and p["page"] == 12)
print(page["text"][:300])